# Fraud modelling and feature imputation

This notebook prepares consumer/transaction/merchant features, imputes missing predictors using training data, and compares log-target OLS, random forest, and XGBoost regression against a training-mean baseline.

Run from top to bottom. Transaction outlier exploration now lives in `outliers.ipynb`; its exploratory $8,000 cutoff is **not applied here**, matching the original workflow. This models the supplied fraud-probability score, not observed fraud events.

Inputs: `transactions_clean` (already carries the per-transaction `consumer_fraud_probability`, attached by day in `process_cleaning.py`), `merchants_clean`, and `consumers_abs` under `project2/data/tables/processed`. No source tables are overwritten. Saved outputs have been cleared.


## 1. Setup


In [ ]:

# Resolve imports by repository contents, regardless of notebook location.
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "scripts" / "process_cleaning.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project repository")
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))


from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, Window, functions as F

# Works when launched from the repository, project2, or notebooks directory.

PROCESSED_PATH = PROJECT_ROOT / "data" / "tables" / "processed"
if not PROCESSED_PATH.is_dir():
    raise FileNotFoundError("Set PROJECT_ROOT to your project2 directory.")
RANDOM_STATE = 42
spark = SparkSession.builder.config("spark.sql.session.timeZone", "Etc/UTC").config("spark.driver.memory", "6g").getOrCreate()

import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, PowerTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import boxcox


## 2. Load and join modelling inputs

`consumer_fraud_probability` (~35K rows, one row per `(user_id, order_date)` with a labelled score) is the join driver here, not `transactions_clean` (~14M rows). Starting from the raw transactions and attaching the daily fraud score, as `process_cleaning.py` does for storage, fans the same probability out across every same-day transaction and inflates the design matrix to one row per transaction instead of one row per label. Instead, `transactions_clean` is joined on `(user_id, order_date)` only to pull in the exact transaction datetime, merchant, and amount for that day — the earliest transaction is kept when a user has more than one on the labelled date — and `merchants_clean`/`consumers_abs` add merchant and demographic context. `toPandas()` collects this ~35K-row table into driver memory.


In [ ]:
consumer_fraud = spark.read.parquet(str(PROCESSED_PATH / "consumer_fraud_probability"))
# clean_fraud_table (process_cleaning.py) does not statistically impute missing
# scores, so a null can still occur even though none are present in this snapshot.
consumer_fraud = (
    consumer_fraud
    .where(F.col("fraud_probability").isNotNull())
    .withColumnRenamed("order_datetime", "order_date")
)

trans = spark.read.parquet(str(PROCESSED_PATH / "transactions_clean"))
# Retain the original deduplication rule; review whether merchant_abn/order ID
# should also be part of this key before treating it as a production rule.
DUPLICATE_KEY = ["order_datetime", "user_id", "dollar_value"]
trans = trans.dropDuplicates(DUPLICATE_KEY)

# consumer_fraud_probability is recorded once per (user_id, order_date), but a
# user can have several transactions that day (true for ~41% of matched pairs
# in this data). Keep one transaction deterministically -- earliest by time --
# so the join below adds exactly one row per label instead of fanning it out,
# but first count the day's transactions so that signal isn't simply dropped.
day_partition = Window.partitionBy("user_id", F.to_date("order_datetime"))
day_order = day_partition.orderBy("order_datetime")
trans_one_per_day = (
    trans
    .withColumn("order_date", F.to_date("order_datetime"))
    .withColumn("transactions_on_day", F.count("*").over(day_partition))
    .withColumn("_rank", F.row_number().over(day_order))
    .where(F.col("_rank") == 1)
    .select("user_id", "order_date", "merchant_abn", "order_datetime",
            "dollar_value", "transactions_on_day")
)

merchants = spark.read.parquet(str(PROCESSED_PATH / "merchants_clean"))
consumers_abs = spark.read.parquet(str(PROCESSED_PATH / "consumers_sa2"))

# merchant_fraud_probability is dropped: this notebook models the
# consumer-level score only, per the intro above.
consumers = (
    consumer_fraud
    .join(trans_one_per_day, on=["user_id", "order_date"], how="left")
    .join(consumers_abs, on="user_id", how="left")
    .join(
        merchants.select(
            "merchant_abn",
            "revenue_code",
            "take_rate",
            "keywords_clean"
        ),
        on="merchant_abn",
        how="left",
    )
)

consumers_pd = consumers.toPandas()
print(f"Collected {len(consumers_pd):,} rows for modelling.")


## 3. Validate the target and hold out evaluation rows

Keep the original 80/20 random row split and log-target OLS. Nonpositive or nonfinite targets cannot be logged and are excluded explicitly. Check the target's documented units before interpreting its bounds.

A row split may place the same consumer in both sets. Use a grouped or temporal split for an evaluation of unseen consumers or future transactions. No feature preprocessing below is fitted to the test set.


In [ ]:
ABS_COLS = ["abs_population", "abs_median_age", "abs_median_personal_income_weekly",
            "abs_median_household_income_weekly", "abs_median_rent_weekly",
            "abs_median_mortgage_repay_monthly"]
CAT_COLS = ["state", "gender", "revenue_code", "order_day_of_week", "order_month"]
target = pd.to_numeric(consumers_pd["fraud_probability"], errors="coerce")
valid_target = target.notna() & np.isfinite(target) & (target > 0)
print("Rows excluded for missing/nonpositive/nonfinite target:", int((~valid_target).sum()))
model_data = consumers_pd.loc[valid_target].copy()
y = target.loc[valid_target]
model_data["abs_matched"] = model_data[ABS_COLS].notna().any(axis=1).astype(int)

# order_datetime is the earliest same-day transaction for this user (see the
# join cell); transactions_on_day, attached in that same join, is the actual
# count of transactions that user made on the labelled date.
model_data["order_datetime"] = pd.to_datetime(model_data["order_datetime"])
model_data["order_day_of_week"] = model_data["order_datetime"].dt.dayofweek.astype(str)
model_data["order_month"] = model_data["order_datetime"].dt.month.astype(str)
model_data["order_is_weekend"] = (model_data["order_datetime"].dt.dayofweek >= 5).astype(int)
model_data["order_days_since_start"] = (
    model_data["order_datetime"] - model_data["order_datetime"].min()
).dt.days

# order_date is the join key from consumer_fraud_probability (date-truncated);
# it's redundant with order_datetime, which already derives the features above.
X = model_data.drop(columns=["fraud_probability", "user_id", "consumer_id",
                            "merchant_abn", "order_datetime", "order_date",
                            "postcode"], errors="ignore")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)
X_train, X_test = X_train.copy(), X_test.copy()


## 4. Training-target exploration

These plots retain the original transform comparisons but use training targets only. Box–Cox and Yeo–Johnson are exploratory; they do not change the models below.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
transforms = {"Original": y_train, "Square root": np.sqrt(y_train),
              "Log": np.log(y_train), "Log1p": np.log1p(y_train)}
if y_train.nunique() > 1:
    transformed, lam = boxcox(y_train)
    transforms[f"Box-Cox (lambda={lam:.3f})"] = transformed
    transforms["Yeo-Johnson"] = PowerTransformer(method="yeo-johnson").fit_transform(
        y_train.to_numpy().reshape(-1, 1)).ravel()
for ax, (label, values) in zip(axes.flat, transforms.items()):
    ax.hist(values, bins=40)
    ax.set(title=label, ylabel="Count")
for ax in list(axes.flat)[len(transforms):]:
    ax.set_visible(False)
fig.tight_layout()
plt.show()


## 5. Fit feature preprocessing on training data


In [ ]:
def keyword_list(value):
    return list(value) if isinstance(value, (list, tuple, np.ndarray)) else []

mlb = MultiLabelBinarizer()
train_keywords = X_train.pop("keywords_clean").map(keyword_list)
test_keywords = X_test.pop("keywords_clean").map(keyword_list)
train_kw = mlb.fit_transform(train_keywords)
# Ignore unseen test keywords without fitting the vocabulary on held-out data.
known_keywords = set(mlb.classes_)
test_kw = mlb.transform(test_keywords.map(lambda values: [v for v in values if v in known_keywords]))
kw_names = [f"kw_{kw}" for kw in mlb.classes_]
X_train = pd.concat([X_train, pd.DataFrame(train_kw, columns=kw_names, index=X_train.index)], axis=1)
X_test = pd.concat([X_test, pd.DataFrame(test_kw, columns=kw_names, index=X_test.index)], axis=1)

abs_means = X_train[ABS_COLS].mean()
for frame in (X_train, X_test):
    frame[ABS_COLS] = frame[ABS_COLS].fillna(abs_means)
    frame["take_rate_missing"] = frame["take_rate"].isna().astype(int)
rate_by_revenue = X_train.groupby("revenue_code")["take_rate"].median()
rate_fallback = X_train["take_rate"].median()
for frame in (X_train, X_test):
    frame["take_rate"] = frame["take_rate"].fillna(
        frame["revenue_code"].map(rate_by_revenue)).fillna(rate_fallback)
    frame[CAT_COLS] = frame[CAT_COLS].fillna("Missing").astype(str)

encoder = OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)
encoded_train = encoder.fit_transform(X_train[CAT_COLS])
encoded_test = encoder.transform(X_test[CAT_COLS])
encoded_names = encoder.get_feature_names_out(CAT_COLS)
def combine_features(frame, encoded):
    numeric = frame.drop(columns=CAT_COLS)
    nonnumeric = numeric.select_dtypes(exclude=[np.number, "bool"]).columns.tolist()
    if nonnumeric:
        raise ValueError(f"Explicitly encode or exclude these predictors: {nonnumeric}")
    return pd.concat([numeric, pd.DataFrame(encoded, columns=encoded_names, index=frame.index)], axis=1).astype(float)

X_train_final = combine_features(X_train, encoded_train).replace([np.inf, -np.inf], np.nan)
X_test_final = combine_features(X_test, encoded_test).replace([np.inf, -np.inf], np.nan)
# Entirely missing training columns have no estimable median; use a fixed zero fallback.
train_medians = X_train_final.median().fillna(0.0)
X_train_final = X_train_final.fillna(train_medians)
X_test_final = X_test_final.fillna(train_medians)
assert X_train_final.columns.equals(X_test_final.columns)
assert np.isfinite(X_train_final.to_numpy()).all()
assert np.isfinite(X_test_final.to_numpy()).all()
print("Training shape:", X_train_final.shape, "Test shape:", X_test_final.shape)


## 6. Shared evaluation

All models are evaluated on the original target scale using the same held-out rows. RMSE includes the square root for both model and baseline. Predictions are not clipped or presented as calibrated probabilities.


In [ ]:
def regression_metrics(actual, predicted):
    return {"RMSE": np.sqrt(mean_squared_error(actual, predicted)),
            "MAE": mean_absolute_error(actual, predicted),
            "R2": r2_score(actual, predicted)}

baseline_pred = np.repeat(y_train.mean(), len(y_test))
baseline_metrics = regression_metrics(y_test, baseline_pred)
evaluations = {"Training-mean baseline": baseline_metrics}
def evaluate(name, predicted):
    metrics = regression_metrics(y_test, predicted)
    for metric in ("RMSE", "MAE"):
        base = baseline_metrics[metric]
        metrics[f"{metric} improvement (%)"] = 100 * (base - metrics[metric]) / base if base else np.nan
    evaluations[name] = metrics
    print(f"{name}: prediction range {np.min(predicted):.4f} to {np.max(predicted):.4f}")
    return pd.DataFrame(evaluations).T


## 7. Log-target OLS

Exponentiation retains the original back-transformation. It is not a bias-corrected conditional mean and does not enforce an upper probability bound.


In [ ]:
X_train_ols = sm.add_constant(X_train_final, has_constant="add")
X_test_ols = sm.add_constant(X_test_final, has_constant="add")
results = sm.OLS(np.log(y_train), X_train_ols).fit()
print(results.summary())
pred_probability = np.exp(results.predict(X_test_ols))
evaluate("Log-target OLS", pred_probability)


## 8. Random forest


In [ ]:
rf = RandomForestRegressor(n_estimators=300, max_depth=None, min_samples_leaf=5,
                           max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_final, y_train)
rf_pred = rf.predict(X_test_final)
evaluate("Random forest", rf_pred)


## 9. XGBoost (optional dependency)


In [ ]:
try:
    from xgboost import XGBRegressor
except ImportError:
    print("XGBoost is not installed; skipping this comparison.")
else:
    xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=5,
                       subsample=0.8, colsample_bytree=0.8,
                       objective="reg:squarederror", random_state=RANDOM_STATE)
    xgb.fit(X_train_final, y_train)
    xgb_pred = xgb.predict(X_test_final)
    evaluate("XGBoost", xgb_pred)


## 10. Model comparison and feature importance


In [ ]:
display(pd.DataFrame(evaluations).T)
rf_importance = pd.DataFrame({"feature": X_train_final.columns,
                              "importance": rf.feature_importances_}).sort_values("importance", ascending=False)
display(rf_importance.head(20))


## Interpretation and next steps

- The design matrix is one row per `consumer_fraud_probability` label (~35K rows). When a user had more than one transaction on the labelled date, `dollar_value`/`order_datetime` describe only the earliest one that day, but `transactions_on_day` retains the true count so that signal isn't lost.
- Feature importance describes model associations, not causal effects.
- Review join multiplicity, target units, and the deduplication key before relying on results.
- Identifiers are excluded, including `merchant_abn`; this corrects its accidental use as a numeric predictor.
- Keyword vocabulary, imputation values, and categorical encoding are now learned from training rows only.
- No model or imputed source table is exported by this notebook.
